In [49]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

In [50]:
population_df = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/master/AnalyseProject/world_population.csv', index_col='Country Code')
population_df.head()

print("=" * 60)
print("STEP 1: DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape: {population_df.shape[0]} rows × {population_df.shape[1]} columns")
print("\n── First 5 Rows ──")
print(population_df.head())
print("\n── Column Data Types ──")
print(population_df.dtypes)
print("\n── Missing Values ──")
missing = population_df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values ✓")

STEP 1: DATASET OVERVIEW

Shape: 217 rows × 58 columns

── First 5 Rows ──
                   1960       1961       1962       1963       1964  \
Country Code                                                          
ABW             54211.0    55438.0    56225.0    56695.0    57032.0   
AFG           8996351.0  9166764.0  9345868.0  9533954.0  9731361.0   
AGO           5643182.0  5753024.0  5866061.0  5980417.0  6093321.0   
ALB           1608800.0  1659800.0  1711319.0  1762621.0  1814135.0   
AND             13411.0    14375.0    15370.0    16412.0    17469.0   

                   1965        1966        1967        1968        1969  ...  \
Country Code                                                             ...   
ABW             57360.0     57715.0     58055.0     58386.0     58726.0  ...   
AFG           9938414.0  10152331.0  10372630.0  10604346.0  10854428.0  ...   
AGO           6203299.0   6309770.0   6414995.0   6523791.0   6642632.0  ...   
ALB           1864791.0   1

In [51]:
print(population_df.columns.tolist())

['1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017']


In [52]:
def get_population_growth_rate_by_country_year(population_df, country_code):
    """
    Computes the population growth rate for a given country from 1961 onwards.

    Parameters
    ----------
    population_df : pd.DataFrame
        DataFrame containing population data with years as columns
        and countries as rows, identified by 'Country Code'.
    country_code : str
        The ISO country code to filter the DataFrame by (e.g., 'ZAF').

    Returns
    -------
    np.ndarray
        A 2D numpy array of shape (n, 2) where the first column contains
        the year and the second column contains the corresponding
        population growth rate, rounded to 5 decimal places.
    """
    country_data = population_df[population_df.index == country_code]
    
    year_cols = [col for col in population_df.columns if str(col).isdigit()]
    populations = country_data[year_cols].values.flatten().astype(float)
    years = list(map(int, year_cols))
    
    growth_rates = []
    for i in range(1, len(populations)):
        rate = (populations[i] - populations[i-1]) / populations[i-1]
        growth_rates.append([years[i], round(rate, 5)])
    
    return np.array(growth_rates)

In [53]:
growth_rate = get_population_growth_rate_by_country_year(population_df,'ABW')

print(growth_rate.shape)


(57, 2)


In [54]:
def feature_response_split(arr):
    """
    Splits a 2D numpy array into training and testing sets based on
    even and odd years respectively.

    Parameters
    ----------
    arr : np.ndarray
        A 2D numpy array of shape (n, 2) where the first column contains
        the year and the second column contains the population growth rate.

    Returns
    -------
    tuple
        Two tuples of the form (X_train, y_train), (X_test, y_test) where:
        - X_train : np.ndarray of even years
        - y_train : np.ndarray of growth rates for even years
        - X_test  : np.ndarray of odd years
        - y_test  : np.ndarray of growth rates for odd years
    """
    # Split into even and odd years using modulo operator
    even_data = arr[arr[:, 0] % 2 == 0]
    odd_data = arr[arr[:, 0] % 2 != 0]

    # First column is the year (X), second column is the growth rate (y)
    X_train, y_train = even_data[:, 0], even_data[:, 1]
    X_test, y_test = odd_data[:, 0], odd_data[:, 1]


    return (X_train, y_train), (X_test, y_test)

In [55]:
data = get_population_growth_rate_by_country_year(population_df,'ABW');
(X_train, y_train), (X_test, y_test) = feature_response_split(data)


In [56]:
def train_model(X_train, y_train, MaxDepth):
    """
    Trains a DecisionTreeRegressor model on the provided training data.

    Parameters
    ----------
    X_train : np.ndarray
        A 1D numpy array containing the training features (even years).
    y_train : np.ndarray
        A 1D numpy array containing the training response (growth rates).
    MaxDepth : int
        The maximum depth of the decision tree.

    Returns
    -------
    DecisionTreeRegressor
        A trained sklearn DecisionTreeRegressor model fitted to the data.
    """
    model = DecisionTreeRegressor(max_depth=MaxDepth)
    model.fit(X_train.reshape(-1, 1), y_train)

    return model

In [68]:
data = get_population_growth_rate_by_country_year(population_df,'ABW')
(X_train, y_train), _ = feature_response_split(data)

train_model(X_train, y_train,3).predict([[2017]])

array([0.00451333])

In [76]:
def test_model(model, y_test, X_test):
    """
    Tests a trained model and returns the Root Mean Squared Logarithmic
    Error (RMSLE) between the predicted and actual values.

    Parameters
    ----------
    model : DecisionTreeRegressor
        A trained sklearn DecisionTreeRegressor model.
    y_test : np.ndarray
        A 1D numpy array containing the actual growth rates for odd years.
    X_test : np.ndarray
        A 1D numpy array containing the testing features (odd years).

    Returns
    -------
    float
        The RMSLE between the predicted and actual values, rounded to
        3 decimal places.
    """
    predictions = model.predict(X_test.reshape(-1, 1))

    rmsle = np.sqrt(np.mean((np.log(1 + predictions) - np.log(1 + y_test)) ** 2))

    return round(float(rmsle), 3)

In [77]:
data = get_population_growth_rate_by_country_year(population_df,'ABW')
(X_train, y_train), (X_test, y_test) = feature_response_split(data)
lm = train_model(X_train, y_train,3)
test_model(lm, y_test, X_test)

0.008

In [75]:
# Check predictions vs actual values
predictions = lm.predict(X_test.reshape(-1, 1))
print("Predictions:", predictions[:5])
print("Actual:", y_test[:5])

Predictions: [0.004563 0.004563 0.004563 0.004563 0.004563]
Actual: [0.02263 0.00836 0.00575 0.00589 0.00582]
